Imports

In [70]:
#libs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import os
import openpyxl
import csv
import sys
import re

In [2]:
csv.field_size_limit(10**7)

# Load in chunks, process each chunk, discard raw text after
results = []
for chunk in pd.read_csv(r"..\data\Newspapers_1.csv", encoding="utf-8", 
                          engine="python", chunksize=500):
    # do your processing here on each chunk, e.g. extract keywords
    results.append(chunk)

news1_df = pd.concat(results, ignore_index=True)

In [3]:
# Load in chunks, process each chunk, discard raw text after
results = []
for chunk in pd.read_csv(r"..\data\Newspapers_2.csv", encoding="utf-8", 
                          engine="python", chunksize=500):
    # do your processing here on each chunk, e.g. extract keywords
    results.append(chunk)

news2_df = pd.concat(results, ignore_index=True)

In [4]:
bonds_df = pd.read_excel(r"..\data\Greece_Germany_Bond_Spread_2010_2026.xlsx")

In [5]:
news1_df.head()

,original_url,final_url,word_count,status,error,date,date_raw,heading_type,article_title,content,extraction_method
0,https://www.capital.gr/diethni/3976815/souidia...,https://www.capital.gr/diethni/3976815/souidia...,148.0,success,NaN,2026-02-26 22:20,"Πέμπτη, 26-Φεβ-2026 22:20",ΔΙΕΘΝΗ,"Drone, πιθανότατα ρωσικό, εντοπίστηκε κοντά στ...",Ένα μη επανδρωμένο αεροσκάφος που εντοπίστηκε ...,readability
1,https://www.capital.gr/diethni/3976810/kouba-o...,https://www.capital.gr/diethni/3976810/kouba-o...,304.0,success,NaN,2026-02-26 21:58,"Πέμπτη, 26-Φεβ-2026 21:58",ΔΙΕΘΝΗ,Ο ένας από τους τέσσερις άνδρες που σκοτώθηκαν...,Ένας από τους τέσσερις άνδρες που σκοτώθηκαν σ...,readability
2,https://www.capital.gr/diethni/3976808/amerika...,https://www.capital.gr/diethni/3976808/amerika...,259.0,success,NaN,2026-02-26 21:39,"Πέμπτη, 26-Φεβ-2026 21:39",ΔΙΕΘΝΗ,Αμερικανοί αξιωματούχοι συναντήθηκαν με τον εγ...,"Αμερικανοί αξιωματούχοι, συνεργάτες του υπουργ...",readability
3,https://www.capital.gr/diethni/3976806/instagr...,https://www.capital.gr/diethni/3976806/instagr...,310.0,success,NaN,2026-02-26 21:28,"Πέμπτη, 26-Φεβ-2026 21:28",ΔΙΕΘΝΗ,Θα ενημερώνει τους γονείς αν το παιδί τους εμφ...,Το Instagram θα προειδοποιεί τους γονείς που έ...,readability
4,https://www.capital.gr/politiki/3976802/k-tsou...,https://www.capital.gr/politiki/3976802/k-tsou...,173.0,success,NaN,2026-02-26 20:58,"Πέμπτη, 26-Φεβ-2026 20:58",ΠΟΛΙΤΙΚΗ,"""Ο κ. Μαρινάκης, πριν ρωτήσει ξανά αν η αντιπο...","Ο εκπρόσωπος Τύπου του ΠΑΣΟΚ-ΚΙΝΑΛ, Κώστας Τσο...",readability


In [37]:
news2_df.head()

,original_url,final_url,word_count,status,error,date,date_raw,heading_type,article_title,content,extraction_method
0,https://www.capital.gr/diethni/3976812/stis-ar...,https://www.capital.gr/diethni/3976812/stis-ar...,143.0,success,NaN,2026-02-26 22:06,"Πέμπτη, 26-Φεβ-2026 22:06",ΔΙΕΘΝΗ,Στις αρχές Μαρτίου στα Εμιράτα η νέα τριμερής ...,Ο επόμενος γύρος των τριμερών ειρηνευτικών δια...,readability
1,https://www.capital.gr/agores/3976809/mikres-m...,https://www.capital.gr/agores/3976809/mikres-m...,282.0,success,NaN,2026-02-26 21:42,"Πέμπτη, 26-Φεβ-2026 21:42",ΑΓΟΡΕΣ,Μικρές μεταβολές ο χρυσός με το βλέμμα στις συ...,Οι τιμές του χρυσού παρέμειναν σχεδόν αμετάβλη...,readability
2,https://www.capital.gr/diethni/3976807/sunanti...,https://www.capital.gr/diethni/3976807/sunanti...,800.0,success,NaN,2026-02-26 21:33,"Πέμπτη, 26-Φεβ-2026 21:33",ΔΙΕΘΝΗ,"Μελόνι: ""Ιταλία και Κύπρος ανήκουν σε έναν...",Ο Κύπριος πρόεδρος Νίκος Χριστοδουλίδης είχε σ...,readability
3,https://www.capital.gr/diethni/3976805/sugkrou...,https://www.capital.gr/diethni/3976805/sugkrou...,224.0,success,NaN,2026-02-26 21:20,"Πέμπτη, 26-Φεβ-2026 21:20",ΔΙΕΘΝΗ,Συγκρούσεις στα σύνορα Πακιστάν - Αφγανιστάν,Οι αρχές των Ταλιμπάν δήλωσαν σήμερα ότι ο αφγ...,readability
4,https://www.capital.gr/diethni/3976801/ipa-ira...,https://www.capital.gr/diethni/3976801/ipa-ira...,220.0,success,NaN,2026-02-26 20:52,"Πέμπτη, 26-Φεβ-2026 20:52",ΔΙΕΘΝΗ,"Ολοκληρώθηκαν οι διαπραγματεύσεις με ""σημαντικ...",Οι διαπραγματεύσεις μεταξύ του Ιράν και των ΗΠ...,readability


In [74]:
bonds_df.head()

,Date,Spread
0,03/01/2026,0.7377
1,02/22/2026,0.6483
2,02/15/2026,0.6095
3,02/08/2026,0.6077
4,02/01/2026,0.6104


# Preprocessing

In [158]:
# Combine both newspaper datasets
df = pd.concat([news1_df, news2_df], ignore_index=True)
# Compute char length
df["char_count"] = df['content'].str.len()

df = df.reset_index(drop=True)
print(f"Total articles: {len(df)}")
df.head()

Total articles: 1566371


,original_url,final_url,word_count,status,error,date,date_raw,heading_type,article_title,content,extraction_method,char_count
0,https://www.capital.gr/diethni/3976815/souidia...,https://www.capital.gr/diethni/3976815/souidia...,148.0,success,NaN,2026-02-26 22:20,"Πέμπτη, 26-Φεβ-2026 22:20",ΔΙΕΘΝΗ,"Drone, πιθανότατα ρωσικό, εντοπίστηκε κοντά στ...",Ένα μη επανδρωμένο αεροσκάφος που εντοπίστηκε ...,readability,1003
1,https://www.capital.gr/diethni/3976810/kouba-o...,https://www.capital.gr/diethni/3976810/kouba-o...,304.0,success,NaN,2026-02-26 21:58,"Πέμπτη, 26-Φεβ-2026 21:58",ΔΙΕΘΝΗ,Ο ένας από τους τέσσερις άνδρες που σκοτώθηκαν...,Ένας από τους τέσσερις άνδρες που σκοτώθηκαν σ...,readability,1978
2,https://www.capital.gr/diethni/3976808/amerika...,https://www.capital.gr/diethni/3976808/amerika...,259.0,success,NaN,2026-02-26 21:39,"Πέμπτη, 26-Φεβ-2026 21:39",ΔΙΕΘΝΗ,Αμερικανοί αξιωματούχοι συναντήθηκαν με τον εγ...,"Αμερικανοί αξιωματούχοι, συνεργάτες του υπουργ...",readability,1800
3,https://www.capital.gr/diethni/3976806/instagr...,https://www.capital.gr/diethni/3976806/instagr...,310.0,success,NaN,2026-02-26 21:28,"Πέμπτη, 26-Φεβ-2026 21:28",ΔΙΕΘΝΗ,Θα ενημερώνει τους γονείς αν το παιδί τους εμφ...,Το Instagram θα προειδοποιεί τους γονείς που έ...,readability,2015
4,https://www.capital.gr/politiki/3976802/k-tsou...,https://www.capital.gr/politiki/3976802/k-tsou...,173.0,success,NaN,2026-02-26 20:58,"Πέμπτη, 26-Φεβ-2026 20:58",ΠΟΛΙΤΙΚΗ,"""Ο κ. Μαρινάκης, πριν ρωτήσει ξανά αν η αντιπο...","Ο εκπρόσωπος Τύπου του ΠΑΣΟΚ-ΚΙΝΑΛ, Κώστας Τσο...",readability,1266


In [159]:
for col in df.columns:
    print(f"{col:20} NaN: {df[col].isna().sum()}")

original_url         NaN: 0
final_url            NaN: 0
word_count           NaN: 382862
status               NaN: 0
error                NaN: 1566371
date                 NaN: 0
date_raw             NaN: 382862
heading_type         NaN: 369298
article_title        NaN: 2
content              NaN: 0
extraction_method    NaN: 0
char_count           NaN: 0


In [160]:
df["status"].value_counts()

status
success    1566371
Name: count, dtype: int64

In [161]:
print(len(df["heading_type"].value_counts()))
print(df["heading_type"].value_counts().head(80).to_string())

164
heading_type
ΔΙΕΘΝΗ                                230510
ΕΠΙΚΑΙΡΟΤΗΤΑ                          221847
VIDEO_NEWS                            177551
ΟΙΚΟΝΟΜΙΑ                             141472
ΕΠΙΧΕΙΡΗΣΕΙΣ                          113014
ΧΡΗΜ. ΑΝΑΚΟΙΝΩΣΕΙΣ                     96891
ΠΟΛΙΤΙΚΗ                               68609
ΑΓΟΡΕΣ                                 53309
CAPITALHEALTH                           7201
BRAND VOICE                             5804
ΜΕ ΑΠΟΨΗ                                5782
ΑΡΘΡΑ                                   5106
ΠΙΣΩ ΑΠΟ ΤΙΣ ΓΡΑΜΜΕΣ                    4906
ΚΩΣΤΑΣ ΣΤΟΥΠΑΣ                          4749
FOREX                                   4566
CAPITALTV                               4010
Ο ΓΙΩΡΓΟΣ ΚΡΑΛΟΓΛΟΥ ΓΡΑΦΕΙ              3077
BLOOMBERG OPINION                       2967
ΠΑΡΟΥΣΙΑΣΕΙΣ                            2864
ΤΕΧΝΟΛΟΓΙΑ                              2541
ΕΡΕΥΝΕΣ                                 2123
REAL ESTATE - ΕΙΔΗΣΕΙΣ                

In [162]:
df = df.drop(columns=['original_url', 'final_url', 'status', 'error', 'date_raw', 'extraction_method'])

In [163]:
df.head()

,word_count,date,heading_type,article_title,content,char_count
0,148.0,2026-02-26 22:20,ΔΙΕΘΝΗ,"Drone, πιθανότατα ρωσικό, εντοπίστηκε κοντά στ...",Ένα μη επανδρωμένο αεροσκάφος που εντοπίστηκε ...,1003
1,304.0,2026-02-26 21:58,ΔΙΕΘΝΗ,Ο ένας από τους τέσσερις άνδρες που σκοτώθηκαν...,Ένας από τους τέσσερις άνδρες που σκοτώθηκαν σ...,1978
2,259.0,2026-02-26 21:39,ΔΙΕΘΝΗ,Αμερικανοί αξιωματούχοι συναντήθηκαν με τον εγ...,"Αμερικανοί αξιωματούχοι, συνεργάτες του υπουργ...",1800
3,310.0,2026-02-26 21:28,ΔΙΕΘΝΗ,Θα ενημερώνει τους γονείς αν το παιδί τους εμφ...,Το Instagram θα προειδοποιεί τους γονείς που έ...,2015
4,173.0,2026-02-26 20:58,ΠΟΛΙΤΙΚΗ,"""Ο κ. Μαρινάκης, πριν ρωτήσει ξανά αν η αντιπο...","Ο εκπρόσωπος Τύπου του ΠΑΣΟΚ-ΚΙΝΑΛ, Κώστας Τσο...",1266


In [164]:
patterns = df['date'].astype(str).apply(lambda x: re.sub(r'\d', '#', x))
print(patterns.value_counts().to_string())

date
####-##-## ##:##       1183934
####-##-## ##:##:##     334997
####-##-##               47391
##:####/##                  49


In [165]:
for pattern, group in df.groupby(df['date'].astype(str).apply(lambda x: re.sub(r'\d', '#', x))):
    print(f"{pattern}  →  {group['date'].iloc[0]}")

####-##-##  →  2002-05-16
####-##-## ##:##  →  2026-02-26 22:20
####-##-## ##:##:##  →  2004-07-02 15:39:00
##:####/##  →  23:5926/02


In [166]:
# Drop malformed
mask_bad = df['date'].astype(str).str.match(r'^\d{2}:\d{4}/\d{2}')
df = df[~mask_bad].reset_index(drop=True)

# Keep only first 10 chars (YYYY-MM-DD)
df['date'] = df['date'].astype(str).str[:10]

There cannot be a date where tehre is 1 and not 01 because the whole date would be less character and with a check we would catch them, since its zero, tehre isnt a value. 

In [168]:
date_parts = df['date'].astype(str).str.extract(r'(\d{4})-(\d{2})-(\d{2})')
date_parts.columns = ['year', 'month', 'day']
date_parts = date_parts.astype(float)

print(f"Year  — min: {date_parts['year'].min():.0f}  max: {date_parts['year'].max():.0f}")
print(f"Month — min: {date_parts['month'].min():.0f}  max: {date_parts['month'].max():.0f}  (valid: 1-12)")
print(f"Day   — min: {date_parts['day'].min():.0f}  max: {date_parts['day'].max():.0f}  (valid: 1-31)")
print(f"\nMonth > 12: {(date_parts['month'] > 12).sum()}")
print(f"Day > 31:   {(date_parts['day'] > 31).sum()}")
print(f"Month == 0: {(date_parts['month'] == 0).sum()}")
print(f"Day == 0:   {(date_parts['day'] == 0).sum()}")

Year  — min: 2000  max: 2026
Month — min: 1  max: 12  (valid: 1-12)
Day   — min: 1  max: 31  (valid: 1-31)

Month > 12: 0
Day > 31:   0
Month == 0: 0
Day == 0:   0


In [171]:
short = df[df['date'].str.len() < 10]['date']
print(f"Dates shorter than 10 chars: {len(short)}")
print(short.value_counts().to_string())

Dates shorter than 10 chars: 0
Series([], )
